# 🧱 Two-Level Stacking Ensemble — XGB + LGB + CatBoost → ANN Meta-Learner
## CARNIVAL Competition | notebook-stack.ipynb

---

### 🏗️ Architecture Overview
```
                    LEVEL 0 — Base Models (Optuna-tuned, 5-fold CV)
                    ┌─────────────────────────────────────────────────┐
   Train data ─────┤  XGBoost ──────► OOF probs  + Test probs (avg)  │
                   │  LightGBM ─────► OOF probs  + Test probs (avg)  │
                   │  CatBoost ─────► OOF probs  + Test probs (avg)  │
                   └───────────────────────┬─────────────────────────┘
                                           │  stack as columns
                               ┌───────────▼───────────┐
                               │  Meta-feature matrix   │
                               │  (n_train × 3 OOF)     │
                               │  + scaled raw features │  ← enriched input
                               └───────────┬───────────┘
                                           │
                    LEVEL 1 — ANN Meta-Learner (5-fold CV on meta-features)
                               ┌───────────▼───────────┐
                               │   Residual MLP         │
                               │   Focal Loss γ=2       │
                               │   Meta-OOF probs       │
                               └───────────┬───────────┘
                                           │
                               F1-Optimal Threshold
                                           │
                               submission_stack_<ts>.csv
```

### 🗺️ Notebook Cells
| # | Cell | Description |
|---|------|-------------|
| 1 | Setup | Imports, seeds, directories |
| 2 | Data | Load + class distribution |
| 3 | Features | Three matrices: tree / CatBoost-df / scaled-nn |
| 4 | CV Utils | Shared OOF/TEST accumulators, threshold search |
| 5 | Optuna XGB | 40-trial Bayesian search |
| 6 | Optuna LGB | 40-trial Bayesian search |
| 7 | Optuna CB | 50-trial Bayesian search |
| 8 | Train Base | Retrain all 3 with best params (5-fold) |
| 9 | Meta-Features | Build L1 input matrix (OOF probs + scaled features) |
| 10 | ANN Meta | Train ANN on meta-features (5-fold) |
| 11 | Compare | Base vs meta performance |
| 12 | Submission | F1-optimal threshold → timestamped CSV |
| 13 | Report | *(Standalone — deletable)* |


In [5]:
!pip install imblearn
!pip install optuna



[notice] A new release of pip available: 22.2.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.2.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



     -------------------------------------- 425.6/425.6 kB 4.5 MB/s eta 0:00:00
     ------------------------------------ 265.9/265.9 kB 605.9 kB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 9.2 MB/s eta 0:00:00
     ---------------------------------------- 80.0/80.0 kB 4.4 MB/s eta 0:00:00
     ------------------------------------- 322.8/322.8 kB 10.1 MB/s eta 0:00:00


---
## Cell 1 — Setup & Imports

In [6]:
# ==============================================================================
# CELL 1: SETUP & IMPORTS
# ==============================================================================
import os, sys, json, time, warnings, random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing   import RobustScaler, LabelEncoder
from sklearn.metrics         import (f1_score, roc_auc_score,
                                     average_precision_score,
                                     confusion_matrix, classification_report)

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool, CatBoostError

from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import optuna
from optuna.samplers import TPESampler
from optuna.pruners  import MedianPruner

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR        = Path('.')
DATA_DIR        = BASE_DIR / 'data'
SUBMISSIONS_DIR = BASE_DIR / 'submissions'
REPORTS_DIR     = BASE_DIR / 'reports'
BEST_PARAMS_DIR = BASE_DIR / 'best_params'
for d in [SUBMISSIONS_DIR, REPORTS_DIR, BEST_PARAMS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Global Config ─────────────────────────────────────────────────────────────
SEED           = 42
N_FOLDS        = 5      # final CV for base models and ANN meta-learner
N_FOLDS_OPT    = 3      # faster CV inside Optuna trials
XGB_TRIALS     = 40
LGB_TRIALS     = 40
CB_TRIALS      = 50
SMOTE_STRATEGY = 0.5   # minority → 50% of majority per fold

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)

set_seed()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Setup complete")
print(f"   Device         : {DEVICE}")
print(f"   XGB trials     : {XGB_TRIALS}  |  LGB trials: {LGB_TRIALS}  |  CB trials: {CB_TRIALS}")
print(f"   Final CV folds : {N_FOLDS}     |  Optuna CV : {N_FOLDS_OPT}")
print(f"   SMOTE strategy : {SMOTE_STRATEGY}")


✅ Setup complete
   Device         : cuda
   XGB trials     : 40  |  LGB trials: 40  |  CB trials: 50
   Final CV folds : 5     |  Optuna CV : 3
   SMOTE strategy : 0.5


---
## Cell 2 — Data Loading

In [8]:
# ==============================================================================
# CELL 2: DATA LOADING
# ==============================================================================
print("⏳ Loading data...")
t0 = time.time()
df_train = pd.read_csv('train.csv', low_memory=False)
df_test  = pd.read_csv('test.csv',  low_memory=False)
df_sub   = pd.read_csv('sample_submission.csv')
print(f"   {time.time()-t0:.1f}s  |  Train={df_train.shape}  Test={df_test.shape}")

TARGET_COL   = 'TARGET'
ID_COL       = 'id'
FEAT_COLS    = [c for c in df_train.columns if c != TARGET_COL]
CAT_COLS     = ['feat_142','feat_157','feat_318','feat_320','feat_325','feat_337']
test_ids     = df_test[ID_COL].values
y            = df_train[TARGET_COL].values
class_counts = np.bincount(y)
SCALE_POS    = class_counts[0] / class_counts[1]

print(f"\n⚖️  Class 0 : {class_counts[0]:,} ({class_counts[0]/len(y)*100:.2f}%)")
print(f"   Class 1 : {class_counts[1]:,} ({class_counts[1]/len(y)*100:.2f}%)")
print(f"   Ratio   : {SCALE_POS:.1f}:1   (scale_pos_weight = {SCALE_POS:.2f})")


⏳ Loading data...
   7.6s  |  Train=(76020, 351)  Test=(60654, 351)

⚖️  Class 0 : 73,012 (96.04%)
   Class 1 : 3,008 (3.96%)
   Ratio   : 24.3:1   (scale_pos_weight = 24.27)


---
## Cell 3 — Feature Engineering

In [9]:
# ==============================================================================
# CELL 3: FEATURE ENGINEERING
# ==============================================================================
# Produces three specialised feature matrices:
#
#   X_tree / X_tree_test    — float64 numpy array  (XGB + LGB)
#   X_cb_df / X_cb_test_df  — pandas DataFrame with string cats  (CatBoost)
#   X_scaled / X_scaled_test— RobustScaler normalised  (ANN meta enrichment)
#
# 13 row-level statistical features appended to all matrices.
# ==============================================================================
set_seed()

X_raw      = df_train[FEAT_COLS].copy()
X_test_raw = df_test[[c for c in FEAT_COLS if c in df_test.columns]].copy()

# Drop zero-variance
zero_var   = [c for c in X_raw.select_dtypes(include=np.number).columns
              if X_raw[c].nunique() <= 1]
X_raw.drop(columns=zero_var, inplace=True)
X_test_raw.drop(columns=[c for c in zero_var if c in X_test_raw.columns], inplace=True)

ACTIVE_NUM = [c for c in X_raw.columns if c not in CAT_COLS]
ACTIVE_CAT = [c for c in CAT_COLS if c in X_raw.columns]
print(f"Active features: {len(ACTIVE_NUM)} numeric | {len(ACTIVE_CAT)} categorical")

# ── 1. CatBoost copy — string cats ────────────────────────────────────────────
X_cb_df      = X_raw.copy()
X_cb_test_df = X_test_raw.copy()
for c in ACTIVE_CAT:
    X_cb_df[c]      = X_cb_df[c].astype(str)
    X_cb_test_df[c] = X_cb_test_df[c].astype(str)

# ── 2. Label encode for XGB / LGB ─────────────────────────────────────────────
for c in ACTIVE_CAT:
    le = LabelEncoder()
    le.fit(pd.concat([X_raw[c], X_test_raw[c]]).astype(str))
    X_raw[c]      = le.transform(X_raw[c].astype(str))
    X_test_raw[c] = le.transform(X_test_raw[c].astype(str))

# ── 3. Smoothed target encoding ───────────────────────────────────────────────
SMOOTH, g_mean = 20, y.mean()
for c in ACTIVE_CAT:
    stats = (pd.DataFrame({'c': X_raw[c], 't': y})
               .groupby('c')['t'].agg(['mean','count']))
    stats['sm'] = ((stats['mean']*stats['count'] + g_mean*SMOOTH)
                   / (stats['count'] + SMOOTH))
    mp  = stats['sm'].to_dict()
    enc = f"{c}_te"
    X_raw[enc]            = X_raw[c].map(mp).fillna(g_mean)
    X_test_raw[enc]       = X_test_raw[c].map(mp).fillna(g_mean)
    X_cb_df[enc]          = X_raw[enc]
    X_cb_test_df[enc]     = X_test_raw[enc]

# ── 4. Row-level statistics (13 features) ─────────────────────────────────────
def add_row_stats(df_x, num_cols):
    arr = df_x[num_cols].values.astype(np.float64)
    df_x['row_mean']      = arr.mean(axis=1)
    df_x['row_std']       = arr.std(axis=1)
    df_x['row_max']       = arr.max(axis=1)
    df_x['row_min']       = arr.min(axis=1)
    df_x['row_sum']       = arr.sum(axis=1)
    df_x['row_nonzero']   = (arr != 0).sum(axis=1)
    df_x['row_range']     = arr.max(axis=1) - arr.min(axis=1)
    df_x['row_skew']      = pd.DataFrame(arr).skew(axis=1).values
    df_x['row_q25']       = np.percentile(arr, 25, axis=1)
    df_x['row_q75']       = np.percentile(arr, 75, axis=1)
    df_x['row_iqr']       = df_x['row_q75'] - df_x['row_q25']
    df_x['row_neg_count'] = (arr < 0).sum(axis=1)
    df_x['row_pos_rate']  = (arr > 0).mean(axis=1)

for df_x in [X_raw, X_test_raw, X_cb_df, X_cb_test_df]:
    add_row_stats(df_x, ACTIVE_NUM)

# ── 5. Final arrays ───────────────────────────────────────────────────────────
X_tree      = X_raw.values.astype(np.float64)
X_tree_test = X_test_raw.values.astype(np.float64)

scaler         = RobustScaler()
X_scaled       = scaler.fit_transform(X_tree)
X_scaled_test  = scaler.transform(X_tree_test)

num_cb = [c for c in X_cb_df.columns if c not in ACTIVE_CAT]
X_cb_df[num_cb]      = X_cb_df[num_cb].astype(np.float32)
X_cb_test_df[num_cb] = X_cb_test_df[num_cb].astype(np.float32)

print(f"\nFeature matrices:")
print(f"  X_tree      : {X_tree.shape}   → XGBoost, LightGBM")
print(f"  X_cb_df     : {X_cb_df.shape}  → CatBoost (native cats)")
print(f"  X_scaled    : {X_scaled.shape}   → ANN meta enrichment")
print("✅ Feature engineering complete.")


Active features: 316 numeric | 6 categorical

Feature matrices:
  X_tree      : (76020, 341)   → XGBoost, LightGBM
  X_cb_df     : (76020, 341)  → CatBoost (native cats)
  X_scaled    : (76020, 341)   → ANN meta enrichment
✅ Feature engineering complete.


---
## Cell 4 — CV Utilities & OOF Accumulators

In [10]:
# ==============================================================================
# CELL 4: CV UTILITIES
# ==============================================================================
set_seed()
SKF     = StratifiedKFold(n_splits=N_FOLDS,     shuffle=True, random_state=SEED)
SKF_OPT = StratifiedKFold(n_splits=N_FOLDS_OPT, shuffle=True, random_state=SEED)

# OOF / TEST accumulators — filled by each base-model training cell
# Shape: OOF[name] → (n_train,)   TEST[name] → (n_test,)
OOF    = {}
TEST   = {}
SCORES = {}


def macro_f1_at(yt, yp, t=0.5):
    return f1_score(yt, (yp >= t).astype(int), average='macro')


def find_best_threshold(yt, yp):
    """Grid search 0.05–0.60 (step=0.005) maximising OOF Macro F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.61, 0.005):
        f1 = macro_f1_at(yt, yp, t)
        if f1 > best_f1:
            best_f1, best_t = f1, round(t, 3)
    return best_t, best_f1


def evaluate(yt, yp, t=0.5, label='', verbose=True):
    pred = (yp >= t).astype(int)
    m = dict(
        macro_f1 = f1_score(yt, pred, average='macro'),
        f1_c1    = f1_score(yt, pred, average='binary', pos_label=1),
        roc_auc  = roc_auc_score(yt, yp),
        pr_auc   = average_precision_score(yt, yp),
        pred_1s  = int(pred.sum()),
    )
    if verbose:
        tag = f'[{label}] ' if label else ''
        print(f"   {tag}Macro F1={m['macro_f1']:.4f}  F1-C1={m['f1_c1']:.4f}  "
              f"ROC-AUC={m['roc_auc']:.4f}  PR-AUC={m['pr_auc']:.4f}  "
              f"pred-1s={m['pred_1s']:,}")
    return m


def smote_fit(X, y):
    sm = SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=SEED, k_neighbors=5)
    return sm.fit_resample(X, y)


def run_tree_cv(cls, params, X, y, X_test, name):
    """Generic sklearn-API CV for XGB/LGB with SMOTE inside each fold."""
    set_seed()
    oof_p = np.zeros(len(y)); test_p = np.zeros(len(X_test)); folds = []
    for fold, (tr, val) in enumerate(SKF.split(X, y), 1):
        Xtr, ytr = smote_fit(X[tr], y[tr])
        m = cls(**params); m.fit(Xtr, ytr)
        vp = m.predict_proba(X[val])[:, 1]
        oof_p[val] = vp
        test_p    += m.predict_proba(X_test)[:, 1] / N_FOLDS
        fi = macro_f1_at(y[val], vp)
        folds.append(fi)
        print(f"   Fold {fold}/{N_FOLDS}  Macro F1={fi:.4f}")
    OOF[name], TEST[name], SCORES[name] = oof_p, test_p, folds
    print(f"   CV: {np.mean(folds):.4f} ± {np.std(folds):.4f}")
    return oof_p, test_p


def run_cb_cv(params, X_df, y, X_test_df, name='catboost'):
    """CatBoost CV with DataFrame Pool + SMOTE on numeric cols."""
    set_seed()
    oof_p     = np.zeros(len(y)); test_p = np.zeros(len(X_test_df)); folds = []
    test_pool = Pool(X_test_df, cat_features=ACTIVE_CAT)
    cat_modes = {c: X_df[c].mode()[0] for c in ACTIVE_CAT}
    num_mask  = [c for c in X_df.columns if c not in ACTIVE_CAT]

    for fold, (tr, val) in enumerate(SKF.split(X_df, y), 1):
        X_tr = X_df.iloc[tr].reset_index(drop=True)
        X_vl = X_df.iloc[val].reset_index(drop=True)
        y_tr, y_vl = y[tr], y[val]

        X_num_s, y_tr_s = smote_fit(X_tr[num_mask].values.astype(np.float64), y_tr)
        n_s   = len(y_tr_s) - len(y_tr)
        X_tr_s = pd.DataFrame(X_num_s, columns=num_mask)
        for c in ACTIVE_CAT:
            X_tr_s[c] = np.concatenate([X_tr[c].values,
                                         np.array([cat_modes[c]] * n_s)])
        X_tr_s = X_tr_s[list(X_tr.columns)]

        cb = CatBoostClassifier(**params)
        cb.fit(Pool(X_tr_s, label=y_tr_s, cat_features=ACTIVE_CAT),
               eval_set=Pool(X_vl, label=y_vl, cat_features=ACTIVE_CAT),
               verbose=0)

        vp = cb.predict_proba(Pool(X_vl, cat_features=ACTIVE_CAT))[:, 1]
        oof_p[val] = vp
        test_p    += cb.predict_proba(test_pool)[:, 1] / N_FOLDS
        fi = macro_f1_at(y_vl, vp)
        folds.append(fi)
        print(f"   Fold {fold}/{N_FOLDS}  Macro F1={fi:.4f}  "
              f"Best iter={cb.get_best_iteration()}")

    OOF[name], TEST[name], SCORES[name] = oof_p, test_p, folds
    print(f"   CV: {np.mean(folds):.4f} ± {np.std(folds):.4f}")
    return oof_p, test_p


print("✅ CV utilities ready.")


✅ CV utilities ready.


---
## Cell 5 — ⚙️ Optuna → XGBoost

In [11]:
# ==============================================================================
# CELL 5: OPTUNA SEARCH — XGBOOST
# ==============================================================================
set_seed()

def xgb_objective(trial):
    p = dict(
        n_estimators      = trial.suggest_int('n_estimators', 300, 1200),
        max_depth         = trial.suggest_int('max_depth', 4, 10),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_weight  = trial.suggest_int('min_child_weight', 1, 10),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        gamma             = trial.suggest_float('gamma', 0, 5),
        scale_pos_weight  = trial.suggest_float('scale_pos_weight',
                                SCALE_POS * 0.8, SCALE_POS * 2.0),
        eval_metric='logloss', random_state=SEED, n_jobs=-1, verbosity=0,
    )
    scores = []
    for step, (tr, val) in enumerate(SKF_OPT.split(X_tree, y)):
        Xtr, ytr = smote_fit(X_tree[tr], y[tr])
        m = xgb.XGBClassifier(**p); m.fit(Xtr, ytr)
        scores.append(macro_f1_at(y[val], m.predict_proba(X_tree[val])[:, 1]))
        trial.report(np.mean(scores), step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(scores)

print(f"🔍 XGBoost Optuna ({XGB_TRIALS} trials)...\n")
t0 = time.time()
xgb_study = optuna.create_study(direction='maximize',
    sampler=TPESampler(seed=SEED, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1))
xgb_study.optimize(xgb_objective, n_trials=XGB_TRIALS, show_progress_bar=True)

XGB_BEST = xgb_study.best_params.copy()
XGB_BEST.update({'eval_metric':'logloss','random_state':SEED,'n_jobs':-1,'verbosity':0})
print(f"\n⏱️  {time.time()-t0:.0f}s | Best OOF Macro F1: {xgb_study.best_value:.4f}")
print("📋 Best XGBoost params:")
for k, v in XGB_BEST.items(): print(f"   {k:<25}: {v}")
with open(BEST_PARAMS_DIR / 'best_params_xgboost.json', 'w') as f:
    json.dump(XGB_BEST, f, indent=2)
print(f"💾 Saved → best_params/best_params_xgboost.json")


🔍 XGBoost Optuna (40 trials)...



  0%|          | 0/40 [00:00<?, ?it/s]


⏱️  4064s | Best OOF Macro F1: 0.7343
📋 Best XGBoost params:
   n_estimators             : 994
   max_depth                : 7
   learning_rate            : 0.05238895962558236
   subsample                : 0.843407356654705
   colsample_bytree         : 0.5497174916395089
   min_child_weight         : 7
   reg_alpha                : 0.08282756807787994
   reg_lambda               : 3.1067410853544954
   gamma                    : 1.8943143399684892
   scale_pos_weight         : 38.60861568202697
   eval_metric              : logloss
   random_state             : 42
   n_jobs                   : -1
   verbosity                : 0
💾 Saved → best_params/best_params_xgboost.json


---
## Cell 6 — ⚙️ Optuna → LightGBM

In [12]:
# ==============================================================================
# CELL 6: OPTUNA SEARCH — LIGHTGBM
# ==============================================================================
set_seed()

def lgb_objective(trial):
    p = dict(
        n_estimators      = trial.suggest_int('n_estimators', 300, 1200),
        num_leaves        = trial.suggest_int('num_leaves', 31, 255),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        min_split_gain    = trial.suggest_float('min_split_gain', 0.0, 1.0),
        is_unbalance=True, random_state=SEED, n_jobs=-1, verbose=-1,
    )
    scores = []
    for step, (tr, val) in enumerate(SKF_OPT.split(X_tree, y)):
        Xtr, ytr = smote_fit(X_tree[tr], y[tr])
        m = lgb.LGBMClassifier(**p); m.fit(Xtr, ytr)
        scores.append(macro_f1_at(y[val], m.predict_proba(X_tree[val])[:, 1]))
        trial.report(np.mean(scores), step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(scores)

print(f"🔍 LightGBM Optuna ({LGB_TRIALS} trials)...\n")
t0 = time.time()
lgb_study = optuna.create_study(direction='maximize',
    sampler=TPESampler(seed=SEED, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1))
lgb_study.optimize(lgb_objective, n_trials=LGB_TRIALS, show_progress_bar=True)

LGB_BEST = lgb_study.best_params.copy()
LGB_BEST.update({'is_unbalance':True,'random_state':SEED,'n_jobs':-1,'verbose':-1})
print(f"\n⏱️  {time.time()-t0:.0f}s | Best OOF Macro F1: {lgb_study.best_value:.4f}")
print("📋 Best LightGBM params:")
for k, v in LGB_BEST.items(): print(f"   {k:<25}: {v}")
with open(BEST_PARAMS_DIR / 'best_params_lightgbm.json', 'w') as f:
    json.dump(LGB_BEST, f, indent=2)
print(f"💾 Saved → best_params/best_params_lightgbm.json")


🔍 LightGBM Optuna (40 trials)...



  0%|          | 0/40 [00:00<?, ?it/s]


⏱️  2894s | Best OOF Macro F1: 0.7376
📋 Best LightGBM params:
   n_estimators             : 1028
   num_leaves               : 41
   learning_rate            : 0.010625811792432868
   subsample                : 0.7098612023990971
   colsample_bytree         : 0.7298901509040301
   min_child_samples        : 61
   reg_alpha                : 0.18383294579853776
   reg_lambda               : 0.02012683662461886
   min_split_gain           : 0.3405146149139783
   is_unbalance             : True
   random_state             : 42
   n_jobs                   : -1
   verbose                  : -1
💾 Saved → best_params/best_params_lightgbm.json


---
## Cell 7 — ⚙️ Optuna → CatBoost

In [15]:
# ==============================================================================
# CELL 7: OPTUNA SEARCH — CATBOOST
# ==============================================================================
set_seed()
cat_modes_opt = {c: X_cb_df[c].mode()[0] for c in ACTIVE_CAT}
num_mask_opt  = [c for c in X_cb_df.columns if c not in ACTIVE_CAT]

def cb_objective(trial):
    # Linked search space: cap memory-heavy knobs so we don't trigger CatBoost
    # 'bad allocation' on Windows (high border_count * depth * iterations
    # explodes RAM on a SMOTE-augmented pool).
    depth       = trial.suggest_int('depth', 4, 8)
    border_count = trial.suggest_int('border_count', 32, 128)
    iters_max   = 1500 if depth <= 6 else (1000 if depth == 7 else 700)
    p = dict(
        iterations          = trial.suggest_int('iterations', 300, iters_max),
        depth               = depth,
        learning_rate       = trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1.0, 15.0),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 2.0),
        random_strength     = trial.suggest_float('random_strength', 0.001, 10.0, log=True),
        border_count        = border_count,
        min_data_in_leaf    = trial.suggest_int('min_data_in_leaf', 1, 30),
        auto_class_weights='Balanced', eval_metric='Logloss',
        early_stopping_rounds=30, random_seed=SEED, verbose=0,
    )
    scores = []
    for step, (tr, val) in enumerate(SKF_OPT.split(X_cb_df, y)):
        X_tr = X_cb_df.iloc[tr].reset_index(drop=True)
        X_vl = X_cb_df.iloc[val].reset_index(drop=True)
        y_tr, y_vl = y[tr], y[val]
        X_num_s, y_s = smote_fit(X_tr[num_mask_opt].values.astype(np.float64), y_tr)
        n_s = len(y_s) - len(y_tr)
        X_s = pd.DataFrame(X_num_s, columns=num_mask_opt)
        for c in ACTIVE_CAT:
            X_s[c] = np.concatenate([X_tr[c].values, [cat_modes_opt[c]]*n_s])
        X_s = X_s[list(X_tr.columns)]
        cb = CatBoostClassifier(**p)
        try:
            cb.fit(Pool(X_s, label=y_s, cat_features=ACTIVE_CAT),
                   eval_set=Pool(X_vl, label=y_vl, cat_features=ACTIVE_CAT), verbose=0)
        except (MemoryError, CatBoostError) as e:
            # OOM / bad-allocation: prune so Optuna moves on (don't kill the study).
            trial.set_user_attr('fit_error', f'{type(e).__name__}: {e}')
            raise optuna.TrialPruned()
        vp = cb.predict_proba(Pool(X_vl, cat_features=ACTIVE_CAT))[:, 1]
        scores.append(macro_f1_at(y_vl, vp))
        trial.report(np.mean(scores), step)   
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(scores)

print(f"🔍 CatBoost Optuna ({CB_TRIALS} trials)...\n")
t0 = time.time()
cb_study = optuna.create_study(direction='maximize',
    sampler=TPESampler(seed=SEED, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1))
cb_study.optimize(cb_objective, n_trials=CB_TRIALS, show_progress_bar=True)

CB_BEST = cb_study.best_params.copy()
CB_BEST.update({'auto_class_weights':'Balanced','eval_metric':'Logloss',
                'early_stopping_rounds':50,'random_seed':SEED,'verbose':0})
print(f"\n⏱️  {time.time()-t0:.0f}s | Best OOF Macro F1: {cb_study.best_value:.4f}")
print("📋 Best CatBoost params:")
for k, v in CB_BEST.items(): print(f"   {k:<25}: {v}")
with open(BEST_PARAMS_DIR / 'best_params_catboost_stack.json', 'w') as f:
    json.dump(CB_BEST, f, indent=2)
print(f"💾 Saved → best_params/best_params_catboost_stack.json")

# Optuna history
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, study, title, col in zip(axes,
        [xgb_study, lgb_study, cb_study],
        ['XGBoost','LightGBM','CatBoost'],
        ['#E74C3C','#2ECC71','#3498DB']):
    df = study.trials_dataframe().dropna(subset=['value'])
    df['best'] = df['value'].cummax()
    ax.scatter(df.index, df['value'], alpha=0.35, s=14, color=col)
    ax.plot(df.index, df['best'], color='black', lw=2, label='Best so far')
    ax.set_title(f'Optuna — {title}', fontweight='bold')
    ax.set_xlabel('Trial'); ax.set_ylabel('OOF Macro F1')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Optuna Optimisation History', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(BASE_DIR / 'stack_optuna_history.png', bbox_inches='tight')
plt.show()


🔍 CatBoost Optuna (50 trials)...



  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-08-12 12:36:30,000] Trial 0 failed with parameters: {'depth': 5, 'border_count': 124, 'iterations': 1179, 'learning_rate': 0.058006322999333615, 'l2_leaf_reg': 3.1842609661941115, 'bagging_temperature': 0.3119890406724053, 'random_strength': 0.0017073967431528124, 'min_data_in_leaf': 26} because of the following error: NameError("name 'CatBoostError' is not defined").
Traceback (most recent call last):
  File "C:\Users\Tanvir Ishrak\AppData\Local\Temp\ipykernel_11472\866841612.py", line 40, in cb_objective
    cb.fit(Pool(X_s, label=y_s, cat_features=ACTIVE_CAT),
  File "c:\Users\Tanvir Ishrak\AppData\Local\Programs\Python\Python310\lib\site-packages\catboost\core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_best_model,
  File "c:\Users\Tanvir Ishrak\AppData\Local\Programs\Python\Python310\lib\site-packages\catboost\core.py", line 2716, in _fit
    self._train(
  Fil

NameError: name 'CatBoostError' is not defined

---
## Cell 8 — 🏋️ Train Base Models with Best Parameters (5-Fold CV)

In [ ]:
# ==============================================================================
# CELL 8: RETRAIN ALL BASE MODELS — FULL 5-FOLD CV
# ==============================================================================
# XGBoost and LightGBM use X_tree (float64 numpy).
# CatBoost uses X_cb_df (pandas DataFrame, string cats, Pool).
# SMOTE 0.5 applied inside each fold for all models.
# OOF + TEST probabilities stored for meta-feature construction in Cell 9.
# ==============================================================================

# ── XGBoost ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("🌲 XGBoost — Retraining with best params")
print("=" * 60)
t0 = time.time()
run_tree_cv(xgb.XGBClassifier, XGB_BEST, X_tree, y, X_tree_test, 'xgboost')
print(f"⏱️  {time.time()-t0:.0f}s | OOF:")
evaluate(y, OOF['xgboost'], t=find_best_threshold(y, OOF['xgboost'])[0], label='XGBoost')

# ── LightGBM ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("💡 LightGBM — Retraining with best params")
print("=" * 60)
t0 = time.time()
run_tree_cv(lgb.LGBMClassifier, LGB_BEST, X_tree, y, X_tree_test, 'lightgbm')
print(f"⏱️  {time.time()-t0:.0f}s | OOF:")
evaluate(y, OOF['lightgbm'], t=find_best_threshold(y, OOF['lightgbm'])[0], label='LightGBM')

# ── CatBoost ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🐱 CatBoost — Retraining with best params")
print("=" * 60)
t0 = time.time()
run_cb_cv(CB_BEST, X_cb_df, y, X_cb_test_df, 'catboost')
print(f"⏱️  {time.time()-t0:.0f}s | OOF:")
evaluate(y, OOF['catboost'], t=find_best_threshold(y, OOF['catboost'])[0], label='CatBoost')

print("\n✅ All base models trained. OOF + TEST probs ready for meta-learner.")


---
## Cell 9 — 🧱 Build Meta-Feature Matrix (L1 Input)

In [ ]:
# ==============================================================================
# CELL 9: BUILD META-FEATURE MATRIX
# ==============================================================================
# The ANN meta-learner trains on a RICHER input than just the 3 OOF probs:
#
#   ┌─────────────────────────────────────────────────────────────────┐
#   │  Column group        │  Dim  │  Contents                        │
#   ├─────────────────────────────────────────────────────────────────┤
#   │  OOF probs           │   3   │  XGB, LGB, CB predictions        │
#   │  OOF rank-norm       │   3   │  rank-normalised versions [0,1]  │
#   │  Pairwise products   │   3   │  XGB×LGB, XGB×CB, LGB×CB        │
#   │  Agreement signal    │   2   │  mean_prob, std_prob (confidence)│
#   │  Scaled features     │ orig  │  full feature matrix (normalised)│
#   └─────────────────────────────────────────────────────────────────┘
#
# Enriching the meta-input with original features lets the ANN discover
# patterns that the base models may have missed or disagree on.
# ==============================================================================

def rank_norm(arr):
    """Normalise each column to [0,1] by rank."""
    out = np.zeros_like(arr, dtype=np.float32)
    for j in range(arr.shape[1]):
        r = arr[:, j].argsort().argsort().astype(np.float32)
        out[:, j] = r / max(len(r) - 1, 1)
    return out


base_names = ['xgboost', 'lightgbm', 'catboost']

# ── Training meta-features ────────────────────────────────────────────────────
oof_stack   = np.column_stack([OOF[n] for n in base_names]).astype(np.float32)
oof_rank    = rank_norm(oof_stack)
oof_xgb_lgb = (oof_stack[:, 0] * oof_stack[:, 1]).reshape(-1, 1)
oof_xgb_cb  = (oof_stack[:, 0] * oof_stack[:, 2]).reshape(-1, 1)
oof_lgb_cb  = (oof_stack[:, 1] * oof_stack[:, 2]).reshape(-1, 1)
oof_mean    = oof_stack.mean(axis=1, keepdims=True)
oof_std     = oof_stack.std(axis=1,  keepdims=True)

META_TRAIN = np.hstack([
    oof_stack,                          # 3  raw OOF probs
    oof_rank,                           # 3  rank-normalised
    oof_xgb_lgb, oof_xgb_cb, oof_lgb_cb,  # 3  pairwise products
    oof_mean, oof_std,                  # 2  ensemble confidence
    X_scaled.astype(np.float32),        # D  original scaled features
])

# ── Test meta-features ────────────────────────────────────────────────────────
test_stack   = np.column_stack([TEST[n] for n in base_names]).astype(np.float32)
test_rank    = rank_norm(test_stack)
test_xgb_lgb = (test_stack[:, 0] * test_stack[:, 1]).reshape(-1, 1)
test_xgb_cb  = (test_stack[:, 0] * test_stack[:, 2]).reshape(-1, 1)
test_lgb_cb  = (test_stack[:, 1] * test_stack[:, 2]).reshape(-1, 1)
test_mean    = test_stack.mean(axis=1, keepdims=True)
test_std     = test_stack.std(axis=1,  keepdims=True)

META_TEST = np.hstack([
    test_stack, test_rank,
    test_xgb_lgb, test_xgb_cb, test_lgb_cb,
    test_mean, test_std,
    X_scaled_test.astype(np.float32),
])

# Normalise the full meta-feature matrix
meta_scaler  = RobustScaler()
META_TRAIN_S = meta_scaler.fit_transform(META_TRAIN).astype(np.float32)
META_TEST_S  = meta_scaler.transform(META_TEST).astype(np.float32)

print(f"Meta-feature matrix (train) : {META_TRAIN_S.shape}")
print(f"Meta-feature matrix (test)  : {META_TEST_S.shape}")
print(f"\nColumn groups:")
print(f"  OOF probs          : 3 cols  ({base_names})")
print(f"  Rank-normalised    : 3 cols")
print(f"  Pairwise products  : 3 cols")
print(f"  Confidence signals : 2 cols  (mean, std)")
print(f"  Scaled features    : {X_scaled.shape[1]} cols")
print(f"  ─────────────────────────────")
print(f"  Total              : {META_TRAIN_S.shape[1]} cols")
print("\n✅ Meta-feature matrix ready.")


---
## Cell 10 — 🧠 ANN Meta-Learner (Level 1)

In [ ]:
# ==============================================================================
# CELL 10: ANN META-LEARNER
# ==============================================================================
# Architecture: Input (meta_dim) → Dense(256)-BN-GELU-Drop
#                               → ResBlock(256)
#                               → Dense(128)-BN-GELU-Drop
#                               → ResBlock(128)
#                               → Dense(64)-BN-GELU-Drop
#                               → Linear(1) → Sigmoid
#
# Loss: Focal Loss (γ=2, α=0.75) — focuses on hard minority examples.
# Optimiser: AdamW + CosineAnnealingWarmRestarts
# Early stopping: patience=15 on OOF Macro F1
# SMOTE 0.5 applied on meta-train inside each fold.
#
# The meta-learner runs its own 5-fold CV to produce meta-OOF predictions
# → stored in OOF['ann_meta'] and TEST['ann_meta'].
# ==============================================================================
set_seed()

class ResBlock(nn.Module):
    def __init__(self, dim, drop=0.3):
        super().__init__()
        self.b   = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(), nn.Dropout(drop),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
    def forward(self, x): return self.act(x + self.b(x))


class StackANN(nn.Module):
    """
    Residual MLP for the stacking meta-learner.
    Uses GELU activations (smoother than ReLU for probability inputs).
    """
    def __init__(self, in_dim, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(drop),
            ResBlock(256, drop),
            nn.Linear(256, 128),    nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(drop),
            ResBlock(128, drop),
            nn.Linear(128, 64),     nn.BatchNorm1d(64),  nn.GELU(), nn.Dropout(drop),
        )
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        return torch.sigmoid(self.head(self.net(x))).squeeze(1)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75, eps=1e-7):
        super().__init__(); self.g, self.a, self.eps = gamma, alpha, eps
    def forward(self, p, t):
        p  = p.clamp(self.eps, 1 - self.eps)
        pt = torch.where(t==1, p, 1-p)
        at = torch.where(t==1,
             torch.full_like(p, self.a), torch.full_like(p, 1-self.a))
        return (-at * (1-pt)**self.g * pt.log()).mean()


def train_ann_fold(Xtr, ytr, Xvl, yvl, Xte,
                   in_dim, epochs=80, bs=512, lr=3e-4):
    set_seed()
    T  = lambda a: torch.tensor(a.astype(np.float32)).to(DEVICE)
    dl = DataLoader(TensorDataset(T(Xtr), T(ytr.astype(np.float32))),
                    batch_size=bs, shuffle=True, drop_last=True)
    model = StackANN(in_dim).to(DEVICE)
    crit  = FocalLoss(gamma=2.0, alpha=0.75)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2)

    best_f1, best_vp, best_tp, patience = 0.0, None, None, 0
    for ep in range(epochs):
        model.train()
        for Xb, yb in dl:
            opt.zero_grad()
            crit(model(Xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vp = model(T(Xvl)).cpu().numpy()
        _, vf1 = find_best_threshold(yvl, vp)
        if vf1 > best_f1:
            best_f1, best_vp, patience = vf1, vp.copy(), 0
            with torch.no_grad():
                best_tp = model(T(Xte)).cpu().numpy()
        else:
            patience += 1
            if patience >= 15: break

    return best_vp, best_tp, best_f1


print("🧠 Training ANN Meta-Learner (Level 1)...\n")
in_dim   = META_TRAIN_S.shape[1]
oof_ann  = np.zeros(len(y))
test_ann = np.zeros(len(META_TEST_S))
ann_f1s  = []
t0 = time.time()

for fold, (tr, val) in enumerate(SKF.split(META_TRAIN_S, y), 1):
    Xtr, ytr = META_TRAIN_S[tr], y[tr]
    Xvl, yvl = META_TRAIN_S[val], y[val]

    # SMOTE on meta-features (all numeric)
    sm = SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=SEED, k_neighbors=5)
    Xtr_s, ytr_s = sm.fit_resample(Xtr, ytr)

    vp, tp, f1 = train_ann_fold(Xtr_s, ytr_s, Xvl, yvl, META_TEST_S, in_dim)
    oof_ann[val] = vp
    test_ann    += tp / N_FOLDS
    ann_f1s.append(f1)
    print(f"   Fold {fold}/{N_FOLDS}  Best Val Macro F1={f1:.4f}")

OOF['ann_meta'], TEST['ann_meta'], SCORES['ann_meta'] = oof_ann, test_ann, ann_f1s
ann_t, ann_f1 = find_best_threshold(y, OOF['ann_meta'])
print(f"\n⏱️  {time.time()-t0:.0f}s | CV: {np.mean(ann_f1s):.4f} ± {np.std(ann_f1s):.4f}")
print(f"   F1-optimal threshold: {ann_t}")
print("\nANN Meta OOF Metrics:")
evaluate(y, OOF['ann_meta'], t=ann_t, label='ANN-Meta')


---
## Cell 11 — 📊 Model Comparison (Base vs Meta)

In [ ]:
# ==============================================================================
# CELL 11: MODEL COMPARISON — BASE MODELS vs ANN META-LEARNER
# ==============================================================================
set_seed()
all_models = ['xgboost','lightgbm','catboost','ann_meta']
colors_map = {'xgboost':'#E74C3C','lightgbm':'#2ECC71',
              'catboost':'#3498DB','ann_meta':'#9B59B6'}

print(f"  {'Model':<14} {'CV Mean':>8} {'CV Std':>8} {'Macro F1':>9} "
      f"{'F1-C1':>7} {'ROC-AUC':>9} {'Thresh':>8} {'Pred-1s':>8}")
print("  " + "─" * 74)

rows = []
for name in all_models:
    oof_p = OOF[name]
    opt_t, _ = find_best_threshold(y, oof_p)
    m = evaluate(y, oof_p, t=opt_t, verbose=False)
    rows.append({**m, 'model':name, 'cv_mean':np.mean(SCORES[name]),
                 'cv_std':np.std(SCORES[name]), 'opt_thresh':opt_t})
    print(f"  {name:<14} {np.mean(SCORES[name]):>8.4f} {np.std(SCORES[name]):>8.4f}"
          f" {m['macro_f1']:>9.4f} {m['f1_c1']:>7.4f} {m['roc_auc']:>9.4f}"
          f" {opt_t:>8.3f} {m['pred_1s']:>8,}")

# Select final model
best_row   = max(rows, key=lambda r: r['macro_f1'])
FINAL_OOF  = OOF[best_row['model']]
FINAL_TEST = TEST[best_row['model']]
FINAL_T    = best_row['opt_thresh']
FINAL_NAME = best_row['model']
print(f"\n✅ Best model: {FINAL_NAME}  (Macro F1 = {best_row['macro_f1']:.4f})")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# 1. Per-fold F1
ax0 = fig.add_subplot(gs[0, :2])
for name in all_models:
    lw = 3 if name == 'ann_meta' else 1.5
    ax0.plot(range(1, N_FOLDS+1), SCORES[name], marker='o', lw=lw,
             color=colors_map[name], label=name)
ax0.set_title('Per-Fold Macro F1 — All Models', fontweight='bold')
ax0.set_xlabel('Fold'); ax0.set_ylabel('Macro F1')
ax0.legend(ncol=2); ax0.grid(True, alpha=0.3)

# 2. OOF Macro F1 bar
ax1 = fig.add_subplot(gs[0, 2])
vals = [r['macro_f1'] for r in rows]
cols = [colors_map[r['model']] for r in rows]
bars = ax1.barh([r['model'] for r in rows], vals, color=cols, edgecolor='k', lw=0.5)
ax1.set_title('OOF Macro F1', fontweight='bold'); ax1.set_xlabel('Macro F1')
ax1.axvline(max(vals), color='red', ls='--', lw=1.2, alpha=0.7)
ax1.grid(True, alpha=0.3, axis='x')

# 3. Score distributions (ANN meta vs best base)
ax2 = fig.add_subplot(gs[1, :2])
for name in ['xgboost','lightgbm','catboost','ann_meta']:
    ax2.hist(OOF[name][y==1], bins=50, alpha=0.45, density=True,
             color=colors_map[name], label=f'{name} (class-1)')
ax2.set_title('OOF Score Distribution — Class-1 Rows Only', fontweight='bold')
ax2.set_xlabel('Predicted Probability'); ax2.set_ylabel('Density')
ax2.legend(ncol=2); ax2.grid(True, alpha=0.3)

# 4. Threshold curve for final model
ax3 = fig.add_subplot(gs[1, 2])
thresholds = np.arange(0.05, 0.65, 0.005)
f1_curve   = [macro_f1_at(y, FINAL_OOF, t) for t in thresholds]
ax3.plot(thresholds, f1_curve, color=colors_map[FINAL_NAME], lw=2)
ax3.fill_between(thresholds, f1_curve, alpha=0.08, color=colors_map[FINAL_NAME])
ax3.axvline(FINAL_T, color='red', ls='--', lw=1.5, label=f'Optimal {FINAL_T}')
ax3.set_title(f'Threshold Curve ({FINAL_NAME})', fontweight='bold')
ax3.set_xlabel('Threshold'); ax3.set_ylabel('Macro F1')
ax3.legend(); ax3.grid(True, alpha=0.3)

plt.suptitle('🧱 Stacking Ensemble — Base vs Meta Analysis', fontweight='bold', fontsize=14)
plt.savefig(BASE_DIR / 'stack_ensemble_analysis.png', bbox_inches='tight', dpi=120)
plt.show()


---
## Cell 12 — 🎯 Submission

In [ ]:
# ==============================================================================
# CELL 12: THRESHOLD OPTIMISATION & SUBMISSION
# ==============================================================================
set_seed()

final_binary = (FINAL_TEST >= FINAL_T).astype(int)
pred_counts  = np.bincount(final_binary)

sub_df = pd.DataFrame({'id': test_ids, 'TARGET': final_binary})
sub_df = df_sub[['id']].merge(sub_df, on='id', how='left')
sub_df['TARGET'] = sub_df['TARGET'].fillna(0).astype(int)

# Format integrity checks
assert list(sub_df.columns) == ['id', 'TARGET'], "Column mismatch!"
assert len(sub_df) == len(df_sub),               "Row count mismatch!"
assert sub_df['TARGET'].isin([0, 1]).all(),       "Non-binary TARGET found!"

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
sub_path  = SUBMISSIONS_DIR / f'submission_stack_{TIMESTAMP}.csv'
sub_df.to_csv(sub_path, index=False)

print(f"🎯 Final model     : {FINAL_NAME}")
print(f"   Threshold       : {FINAL_T:.3f}  (F1-optimal)")
print(f"   Class 0         : {pred_counts[0]:,} ({pred_counts[0]/len(final_binary)*100:.2f}%)")
print(f"   Class 1         : {pred_counts[1]:,} ({pred_counts[1]/len(final_binary)*100:.2f}%)")
print(f"\n✅ All checks passed.")
print(f"💾 Saved: {sub_path}")

SUBMISSION_META = {
    'timestamp'    : TIMESTAMP,
    'filename'     : str(sub_path),
    'final_model'  : FINAL_NAME,
    'threshold'    : float(FINAL_T),
    'pred_class_0' : int(pred_counts[0]),
    'pred_class_1' : int(pred_counts[1]),
    'model_scores' : {n: {'cv_mean': float(np.mean(SCORES[n])),
                          'cv_std' : float(np.std(SCORES[n]))}
                      for n in all_models},
    'xgb_best'     : XGB_BEST,
    'lgb_best'     : LGB_BEST,
    'cb_best'      : CB_BEST,
}


---
## Cell 13 — 📄 Report Generation
> **⚠️ STANDALONE — run after Cell 12. Safe to delete.**

In [ ]:
# ==============================================================================
# CELL 13: REPORT GENERATION  [STANDALONE — DELETABLE]
# ==============================================================================
report_ts   = datetime.now().strftime('%Y%m%d_%H%M%S')
report_path = REPORTS_DIR / f'report_stack_{report_ts}.txt'

fp  = (FINAL_OOF >= FINAL_T).astype(int)
m   = evaluate(y, FINAL_OOF, t=FINAL_T, verbose=False)
cm  = confusion_matrix(y, fp)
cr  = classification_report(y, fp, target_names=['Stable','At-Risk'])
SEP, THIN = '='*70, '-'*70

lines = [
    SEP,
    '  CARNIVAL — Stacking Ensemble Report (notebook-stack.ipynb)',
    SEP,
    f'  Generated  : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    f'  Submission : {SUBMISSION_META["filename"]}',
    '',
    THIN, '  ARCHITECTURE', THIN,
    '  Level 0 base models : XGBoost | LightGBM | CatBoost (Optuna-tuned)',
    '  Level 1 meta-learner: ANN Residual MLP (Focal Loss γ=2, α=0.75)',
    '  Meta-features       : 3 OOF probs + 3 rank-norm + 3 pairwise +',
    '                        2 confidence + full scaled feature matrix',
    f'  SMOTE strategy      : {SMOTE_STRATEGY}  (per fold, minority→50% of majority)',
    '',
    THIN, '  OPTUNA SEARCH', THIN,
    f'  XGBoost  : {XGB_TRIALS} trials | LightGBM: {LGB_TRIALS} trials | CatBoost: {CB_TRIALS} trials',
    f'  Optuna CV : {N_FOLDS_OPT}-fold  |  Final CV: {N_FOLDS}-fold',
    '',
    THIN, '  MODEL PERFORMANCE (OOF, 5-FOLD)', THIN,
    f'  {"Model":<14} {"CV Mean":>9} {"CV Std":>8}',
    '  ' + '-'*35,
] + [f'  {n:<14} {s["cv_mean"]:>9.4f} {s["cv_std"]:>8.4f}'
     for n, s in SUBMISSION_META['model_scores'].items()] + [
    '',
    THIN, '  FINAL MODEL & THRESHOLD', THIN,
    f'  Model     : {SUBMISSION_META["final_model"]}',
    f'  Threshold : {SUBMISSION_META["threshold"]:.3f}  (F1-optimal)',
    f'  Macro F1  : {m["macro_f1"]:.4f}',
    f'  ROC-AUC   : {m["roc_auc"]:.4f}',
    f'  PR-AUC    : {m["pr_auc"]:.4f}',
    '',
    '  Confusion Matrix (OOF):',
    f'    True\\Pred  |  Stable   | At-Risk',
    f'    Stable     | {cm[0,0]:9,} | {cm[0,1]:7,}',
    f'    At-Risk    | {cm[1,0]:9,} | {cm[1,1]:7,}',
    '',
    '  Classification Report:',
] + [f'  {l}' for l in cr.split('\n')] + [
    '',
    THIN, '  SUBMISSION', THIN,
    f'  Class 0  : {SUBMISSION_META["pred_class_0"]:,}',
    f'  Class 1  : {SUBMISSION_META["pred_class_1"]:,}',
    '',
    THIN, '  FUTURE IMPROVEMENTS', THIN,
    '  • Optuna for ANN meta (learning_rate, dropout, hidden dims)',
    '  • Add Random Forest as a 4th base model for more diversity',
    '  • Calibrate base model probabilities before stacking (isotonic)',
    '  • Nelder-Mead weight optimisation over base predictions',
    '  • Deeper stacking: 3 levels (base → lgbm meta → ann top)',
    '',
    SEP, '  END OF REPORT', SEP,
]
text = '\n'.join(lines)
with open(report_path, 'w') as f:
    f.write(text)
print(text)
print(f"\n💾 Report saved: {report_path}")
